# Task 3: Broadcast Join Experiment for Query 3

This experiment joins the original Silver taxi-trips and air-quality Delta tables.

## Connect to Spark

In [1]:
import sys

from pyspark.sql import SparkSession

JOB_PATH = '/opt/spark/jobs'
if JOB_PATH not in sys.path:
    sys.path.insert(0, JOB_PATH)

from analytics.benchmarking import measure_registered_query, rows_match
from analytics.runner import execute_registered_query
from analytics.source_tables import register_broadcast_join_sources

spark = (
    SparkSession.builder
    .master('spark://spark-master:7077')
    .appName('Assignment2-BroadcastJoin-Query3')
    .getOrCreate()
)
spark.conf.set('spark.sql.session.timeZone', 'UTC')
spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/17 18:01:00 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## Register the original Delta tables

In [2]:
spark.catalog.clearCache()
register_broadcast_join_sources(spark)
print('Registered Silver taxi trips and air quality views.')

26/09/17 18:01:02 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.


Registered Silver taxi trips and air quality views.


## Isolate the join strategy

- Automatic broadcasting is disabled
- AQE is also disabled so it cannot change the join at [runtime](https://spark.apache.org/docs/4.0.4/sql-performance-tuning.html#adaptive-query-execution).
- The forced query uses a broadcast hint.

In [3]:
JOIN_CONFIGS = {
    'spark.sql.adaptive.enabled': 'false',
    'spark.sql.autoBroadcastJoinThreshold': '-1',
}
for key, value in JOIN_CONFIGS.items():
    spark.conf.set(key, value)

JOIN_CONFIGS

{'spark.sql.adaptive.enabled': 'false',
 'spark.sql.autoBroadcastJoinThreshold': '-1'}

## Measure normal and broadcast joins

In [4]:
NORMAL_QUERY = 'broadcast_join_normal'
BROADCAST_QUERY = 'broadcast_join_forced'
REPETITIONS = 3

normal_results = measure_registered_query(spark, NORMAL_QUERY, REPETITIONS)
broadcast_results = measure_registered_query(spark, BROADCAST_QUERY, REPETITIONS)

assert rows_match(
    normal_results['rows'],
    broadcast_results['rows'],
), 'Normal and broadcast results differ.'

print('Normal and broadcast results match within float tolerance.')

26/09/17 18:01:03 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Normal and broadcast results match within float tolerance.


## Normal-join physical plan

This indicates normal join was used: `SortMergeJoin`.

In [5]:
execute_registered_query(spark, NORMAL_QUERY).explain('formatted')

== Physical Plan ==
* HashAggregate (20)
+- Exchange (19)
   +- * HashAggregate (18)
      +- * HashAggregate (17)
         +- * HashAggregate (16)
            +- * Project (15)
               +- * SortMergeJoin Inner (14)
                  :- * Sort (5)
                  :  +- Exchange (4)
                  :     +- * Project (3)
                  :        +- * ColumnarToRow (2)
                  :           +- Scan parquet  (1)
                  +- * Sort (13)
                     +- * HashAggregate (12)
                        +- Exchange (11)
                           +- * HashAggregate (10)
                              +- * Project (9)
                                 +- * Filter (8)
                                    +- * ColumnarToRow (7)
                                       +- Scan parquet  (6)


(1) Scan parquet 
Output [2]: [pickup_datetime#4304, pickup_date#4322]
Batched: true
Location: PreparedDeltaFileIndex [s3a://delta-data/silver/silver_taxi_trips]
ReadSchema: struc

## Broadcast-join physical plan

- These indicate broadcast join was used: `BroadcastHashJoin` and `BroadcastExchange`.

In [6]:
execute_registered_query(spark, BROADCAST_QUERY).explain('formatted')

== Physical Plan ==
* HashAggregate (19)
+- Exchange (18)
   +- * HashAggregate (17)
      +- * HashAggregate (16)
         +- Exchange (15)
            +- * HashAggregate (14)
               +- * Project (13)
                  +- * BroadcastHashJoin Inner BuildRight (12)
                     :- * Project (3)
                     :  +- * ColumnarToRow (2)
                     :     +- Scan parquet  (1)
                     +- BroadcastExchange (11)
                        +- * HashAggregate (10)
                           +- Exchange (9)
                              +- * HashAggregate (8)
                                 +- * Project (7)
                                    +- * Filter (6)
                                       +- * ColumnarToRow (5)
                                          +- Scan parquet  (4)


(1) Scan parquet 
Output [2]: [pickup_datetime#4754, pickup_date#4772]
Batched: true
Location: PreparedDeltaFileIndex [s3a://delta-data/silver/silver_taxi_trips]
ReadSchema: 

## Benchmark summary

In [9]:
summary = {
    'normal_samples_seconds': [round(value, 3) for value in normal_results['samples_seconds']],
    'normal_median_seconds': round(normal_results['median_seconds'], 3),
    'broadcast_samples_seconds': [round(value, 3) for value in broadcast_results['samples_seconds']],
    'broadcast_median_seconds': round(broadcast_results['median_seconds'], 3),
    'broadcast_speedup_factor': round(normal_results['median_seconds'] / broadcast_results['median_seconds'],2),
}
summary

{'normal_samples_seconds': [3.872, 2.912, 2.802],
 'normal_median_seconds': 2.912,
 'broadcast_samples_seconds': [2.335, 1.953, 1.881],
 'broadcast_median_seconds': 1.953,
 'broadcast_speedup_factor': 1.49}

## Cleanup

In [10]:
spark.catalog.clearCache()
spark.stop()